# Silver Layer — Products Transformation

Transforms `bronze.products` into `silver.products`. Filters nulls, deduplicates by `product_id`, cleans product_category_name. Additionally, Transforms `bronze.products_category` by cleaning product_category_name. Finally, Join both transformed dataframe using product_category_name column.

Validates three DQ checks.
 
Write transformed join dataframe to silver schema.

## Design Decisions

* **Left join with fallback** — `coalesce(english_name, portuguese_name, 'Unknown')` ensures every product has a usable category name, even for the 3 categories without translations.
* **No value-range checks for product dimensions** — columns like product_weight_g have legitimate nulls; rigorous validation would require a null-tolerant check, deferred for future work.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
%run ../Utilities/silver_dq_checks

# Silver Layer — Data Quality Checks

This notebook defines reusable data quality check functions used across the Silver layer transformation notebooks.

Each function validates a specific property of a transformed DataFrame before it is written to a Silver Delta table. On success the function prints a PASSED message; on failure it raises a `ValueError` with details, stopping the pipeline.

## Checks Defined Here

* `check_not_null` — verifies that specified columns have no null or blank values
* `check_unique` — verifies that the given key (single or composite) has no duplicates
* `check_row_count` — verifies that the Silver row count falls within an expected percentage range of the Bronze source

## How To Use

Import these functions into a Silver transformation notebook by running:
​```
%run ../Utilities/silver_dq_checks
​```
Then call the functions on the transformed DataFrame before writing to Silver.

### check_not_null
Validates that the specified columns contain no null or blank/whitespace values.

### check_unique
Validates that the specified key columns produce unique rows. Supports single-column or composite keys.

### check_row_count
Validates that the Silver row count is within an acceptable percentage range of the Bronze source row count.


### check_event_sequence
Validates that timestamp columns appear in the expected chronological order. Skips rows where either timestamp in a pair is null. Raises on any violation.

### identify_event_sequence_violations
Adds one boolean column per consecutive timestamp pair to mark whether each row violates the expected order. Returns the annotated DataFrame without raising — callers use it to split clean rows from violations.

### check_value_range
Validates that values in specified columns meet a minimum threshold rule (e.g. `price > 0`, `freight_value >= 0`). Each rule is a dict specifying column, min_value, and inclusivity. Raises on any violation, reporting all rules that failed.

In [0]:
catalog_name = 'retail_lakehouse'
bronze_schema = 'bronze'
silver_schema = 'silver'
quarantine_schema = 'quarantine'

%md
### Transformations - Products

In [0]:
df_products = spark.read.table(f'{catalog_name}.{bronze_schema}.products')
transformed_products_df = df_products.filter(col('product_id').isNotNull())\
    .dropDuplicates(['product_id'])\
        .withColumn('product_category_name',lower(trim(col('product_category_name'))))\
        .withColumn('silver_processed_at',current_timestamp())

### Transformations - Product_Category


In [0]:
df_category = spark.read.table(f'{catalog_name}.{bronze_schema}.products_category') \
    .select('product_category_name', 'product_category_name_english')
transformed_category_df = df_category.withColumn(
    'product_category_name', lower(trim(col('product_category_name')))
)

### Joining the transformed Products and Category dataframe

In [0]:
transformed_products_category_df = transformed_products_df.join(transformed_category_df,'product_category_name','left')\
    .withColumn('category_name',coalesce('product_category_name_english','product_category_name',lit('Unknown')))\
        .drop('product_category_name_english')

product_category_name,product_id,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,ingestion_time,source_file,silver_processed_at,category_name
ferramentas_jardim,e1d1d22e9f8122a4ec1533b032c12562,49,1163,9,2150,70,8,34,2026-05-28T09:00:29.750Z,olist_products_dataset.csv,2026-06-05T19:41:48.200Z,garden_tools
esporte_lazer,ce5b91848b91118daffb3af53b747475,50,699,4,1388,34,9,31,2026-05-28T09:00:29.750Z,olist_products_dataset.csv,2026-06-05T19:41:48.200Z,sports_leisure
brinquedos,1c6fb703c624b381a20f21f757694866,33,471,2,725,22,17,15,2026-05-28T09:00:29.750Z,olist_products_dataset.csv,2026-06-05T19:41:48.200Z,toys
papelaria,4e04ffb7dd3739ecfc37de8927dd586c,59,144,1,500,24,7,16,2026-05-28T09:00:29.750Z,olist_products_dataset.csv,2026-06-05T19:41:48.200Z,stationery
ferramentas_jardim,0992c6cba95a13bfa68ea7d5e22d478b,42,671,1,3850,29,12,24,2026-05-28T09:00:29.750Z,olist_products_dataset.csv,2026-06-05T19:41:48.200Z,garden_tools
cama_mesa_banho,32ee3f4e2493045726807808d7abbab6,54,381,2,1400,38,9,27,2026-05-28T09:00:29.750Z,olist_products_dataset.csv,2026-06-05T19:41:48.200Z,bed_bath_table
moveis_decoracao,af16005fca813272caf59c432153949e,55,1186,2,800,53,8,20,2026-05-28T09:00:29.750Z,olist_products_dataset.csv,2026-06-05T19:41:48.200Z,furniture_decor
moveis_decoracao,b2b3503e675abbc07e278151da072ed2,46,287,1,8500,75,25,55,2026-05-28T09:00:29.750Z,olist_products_dataset.csv,2026-06-05T19:41:48.200Z,furniture_decor
fashion_bolsas_e_acessorios,acd427ee119d5c71c7a85ae488cb0a6a,52,68,7,200,16,2,11,2026-05-28T09:00:29.750Z,olist_products_dataset.csv,2026-06-05T19:41:48.200Z,fashion_bags_accessories
consoles_games,51fed4afb1b41d00ad9d25a73b7fbbc7,48,303,3,250,21,7,15,2026-05-28T09:00:29.750Z,olist_products_dataset.csv,2026-06-05T19:41:48.200Z,consoles_games


### Data Quality Checks

In [0]:
check_not_null(transformed_products_category_df, ['product_id'])
check_unique(transformed_products_category_df,['product_id'])
check_row_count(transformed_products_category_df,f'{catalog_name}.{bronze_schema}.products',85,100)

check_not_null: PASSED
check_unique: PASSED
check_row_count: PASSED


## Write to silver schema

In [0]:
transformed_products_category_df.write.format('delta').mode('overwrite').saveAsTable(f'{catalog_name}.{silver_schema}.products')